In [6]:
!python3 -m venv venv
!venv/bin/pip install -r requirements.txt

Error: Command '['/content/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: venv/bin/pip: No such file or directory


In [8]:
!pip install scrapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.4/350.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 6.9 MB/s eta 0:00:00


In [12]:
import scrapy
import os
import requests # Import the requests library

# The URL is a web address, not a local file path
url = "https://www.worldometers.info/gdp/gdp-by-country/"

# Fetch the content from the URL using requests
try:
  response_from_requests = requests.get(url)
  response_from_requests.raise_for_status() # Raise an exception for HTTP errors
  url_data = response_from_requests.text

  # Create a Scrapy TextResponse object
  response = scrapy.http.TextResponse(url=url, body=url_data, encoding='utf-8')
  print(response)
except requests.exceptions.RequestException as e:
  print(f"Error fetching URL: {e}")


<200 https://www.worldometers.info/gdp/gdp-by-country/>


In [20]:
table = response.xpath('//table')[1]

row = table.xpath('.//tr')[1]

print(
    row.xpath('.//text()').getall()
)

[' ', ' 1 ', ' United States ', ' ', '$29.18 trillion', ' ', ' $29,184,890,000,000 ', ' 2.80% ', ' $85,810 ', ' 26.23% ', ' ']


In [24]:
table = response.xpath('//table')[1]

for row in table.xpath('.//tr'):

    data = row.xpath(
        './/text()'
    ).getall()

    data = [
        x.strip()
        for x in data
        if x.strip()
    ]


    if len(data) >= 3:

        country = data[1]

        gdp = data[2]


        print(
            country,
            gdp
        )

Country GDP
United States $29.18 trillion
China $18.74 trillion
Germany $4.66 trillion
Japan $4.03 trillion
India $3.91 trillion
United Kingdom $3.64 trillion
France $3.16 trillion
Italy $2.37 trillion
Canada $2.24 trillion
Brazil $2.18 trillion
Russia $2.17 trillion
South Korea $1.88 trillion
Mexico $1.85 trillion
Australia $1.75 trillion
Spain $1.72 trillion
Indonesia $1.4 trillion
Turkey $1.32 trillion
Saudi Arabia $1.24 trillion
Netherlands $1.23 trillion
Switzerland $936.56 billion
Poland $914.7 billion
Taiwan $801.5 billion
Belgium $664.56 billion
Argentina $633.27 billion
Sweden $610.12 billion
Ireland $577.39 billion
Singapore $547.39 billion
Israel $540.38 billion
United Arab Emirates $537.08 billion
Thailand $526.41 billion
Austria $521.64 billion
Norway $483.73 billion
Vietnam $476.39 billion
Philippines $461.62 billion
Bangladesh $450.12 billion
Iran $436.91 billion
Denmark $429.46 billion
Malaysia $421.97 billion
Colombia $418.54 billion
Hong Kong $407.11 billion
South Afr

In [27]:
import json

with open("gdp-by-country","w") as _f:
  json.dump(data,_f)

In [28]:
column_names = ["Country", "GDP"]

rows = []

for tr in table.xpath('tr'):

    data = tr.xpath(
        './/text()'
    ).getall()


    data = [
        x.strip()
        for x in data
        if x.strip()
    ]


    if len(data) >= 3:

        country = data[1]

        gdp = data[2]


        rows.append(
            [
                country,
                gdp
            ]
        )


rows


[]

In [29]:
# Now persist it to disk
import csv


with open("countries_gdp.csv", "w") as _f:

    writer = csv.writer(_f)


    # write column names

    writer.writerow(
        column_names
    )


    # write scraped rows

    writer.writerows(
        rows
    )

In [30]:
import sqlite3
connection = sqlite3.connect("countries_gdp.db")
db_table = 'CREATE TABLE results (id integer primary key, country TEXT, gdp TEXT)'
cursor = connection.cursor()
cursor.execute(db_table)
connection.commit()

In [34]:
import sqlite3
import pandas as pd


connection = sqlite3.connect("countries_gdp.db")

cursor = connection.cursor()


# remove old data
cursor.execute(
    "DROP TABLE IF EXISTS results"
)


# create table

cursor.execute(
    """
    CREATE TABLE results(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        country TEXT,
        gdp TEXT
    )
    """
)


query = """
INSERT INTO results(country, gdp)
VALUES (?,?)
"""


for tr in table.xpath('.//tr'):

    data = tr.xpath(
        './/text()'
    ).getall()


    data = [
        x.strip()
        for x in data
        if x.strip()
    ]


    if len(data) >= 3:


        country = data[1]

        gdp = data[2]


        # skip header

        if country != "Country":

            cursor.execute(
                query,
                (
                    country,
                    gdp
                )
            )


connection.commit()



# Show final database table

df = pd.read_sql(
    "SELECT * FROM results",
    connection
)


print(df)


connection.close()

      id           country              gdp
0      1     United States  $29.18 trillion
1      2             China  $18.74 trillion
2      3           Germany   $4.66 trillion
3      4             Japan   $4.03 trillion
4      5             India   $3.91 trillion
..   ...               ...              ...
213  214          Kiribati  $307.86 million
214  215  Marshall Islands  $280.36 million
215  216             Nauru  $160.35 million
216  217        Montserrat   $80.43 million
217  218            Tuvalu      $56 million

[218 rows x 3 columns]


0
